In [1]:
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

#####################################
# Chapter 2: 데이터 로딩 및 처리
#####################################

class GPTDatasetV1(Dataset):
    """
    GPT 학습을 위한 데이터셋 클래스입니다.
    텍스트를 입력받아 토큰화하고, 입력(input)과 타겟(target) 쌍을 만듭니다.
    """
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # 1. 전체 텍스트를 토큰화합니다.
        # <|endoftext|> 같은 특수 토큰도 허용하여 인코딩합니다.
        token_ids = ???????

        # 2. 슬라이딩 윈도우 방식으로 데이터를 조각냅니다.
        for i in range(???????, ???????, ???????):
            input_chunk = ???????
            target_chunk = ???????
            
            # 텐서로 변환하여 저장
            self.input_ids.???????
            self.target_ids.???????

    def __len__(self):
        # 데이터셋의 총 샘플 수 반환
        return len(self.input_ids)

    def __getitem__(self, idx):
        # 특정 인덱스의 입력과 타겟 쌍 반환
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True, num_workers=0):
    """
    텍스트 데이터를 받아 학습에 사용할 DataLoader를 생성하는 함수입니다.
    """
    # 토크나이저 초기화 (GPT-2용 BPE 인코딩 사용)
    tokenizer = ???????

    # 데이터셋 생성
    dataset = ???????

    # 데이터로더 생성 (배치 단위로 데이터를 묶어주고 셔플링 수행)
    dataloader = ???????

    return dataloader


In [2]:

#####################################
# Chapter 3: 어텐션 메커니즘 (모델의 핵심)
#####################################
class MultiHeadAttention(nn.Module):
    """
    멀티 헤드 셀프 어텐션 (Multi-Head Self-Attention) 모듈입니다.
    입력 데이터 간의 관계성을 여러 관점(Head)에서 병렬로 학습합니다.
    """
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "출력 차원(d_out)은 헤드 수(num_heads)로 나누어 떨어져야 합니다."

        self.d_out = ???????
        self.num_heads = ???????
        self.head_dim = ???????  # 각 헤드가 담당할 차원 크기

        # Query, Key, Value를 만들기 위한 선형 투영 레이어들
        self.W_query = ???????
        self.W_key = ???????
        self.W_value = ???????
        
        # 멀티 헤드 결과를 하나로 합친 후 통과시키는 출력 레이어
        self.out_proj = ??????? 
        self.dropout = ???????
        
        # Causal Mask (인과적 마스킹) 생성: 미래의 토큰을 보지 못하게 함
        ???????

    def forward(self, x):
        b, num_tokens, d_in = x.shape # b: 배치 크기, num_tokens: 시퀀스 길이

        # 1. Q, K, V 계산
        keys = ???????
        queries = ???????
        values = ???????

        # 2. 헤드 나누기 (Multi-head splitting)
        # 차원을 변형하여 여러 헤드가 병렬로 처리할 수 있게 함
        keys = ???????
        values = ???????
        queries = ???????

        # 3. 차원 순서 변경 (Transpose)
        keys = ???????
        queries = ???????
        values = ???????

        # 4. Scaled Dot-Product Attention 계산
        attn_scores = ???????

        # 5. 마스킹 (Masking)
        mask_bool = ???????
        attn_scores.???????

        # 6. 소프트맥스 및 드롭아웃
        attn_weights = ???????
        attn_weights = ???????

        # 7. Value와의 가중치 합 (Context Vector 계산)
        context_vec = ??????? 

        # 8. 헤드 결합 (Concatenation)
        # 나눠졌던 헤드들을 다시 원래의 d_out 차원으로 합침
        context_vec = ???????
        
        # 9. 최종 선형 투영
        context_vec = ???????

        return context_vec


In [3]:

#####################################
# Chapter 4: GPT 아키텍처 구성 요소
#####################################
class LayerNorm(nn.Module):
    """
    층 정규화 (Layer Normalization): 학습 안정성을 높임
    """
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = ???????
        self.scale = ???????
        self.shift = ???????

    def forward(self, x):
        mean = ???????
        var = ??????? # 분산
        # TODO: LayerNorm 계산식의 통계량 변수를 채우세요.
        norm_x =???????
        # TODO: LayerNorm 학습 파라미터를 채우세요.
        return ???????


class GELU(nn.Module):
    """
    GELU 활성화 함수: GPT 계열에서 주로 사용하는 비선형 함수
    """
    # 활성화함수: 모델에 비선형성을 부여하는 역할. 없다면, 층을 깊게 쌓아도 결국 하나의 거대한 선형 연산에 불과함
    #  어텐션은 단어들 사이의 관계 파악에 집중
    #  피드포워드네트워크는 단어 자체가 가진 의미를 더 깊게 추출 및 변환
    def __init__(self):
        super().__init__()

    def forward(self, x):  #GELU를 채우는 것은 시험에는 안나올 듯?
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    """
    피드 포워드 네트워크 (Feed-Forward Network)
    어텐션이 모은 정보를 각 토큰별로 개별적으로 가공하는 역할
    보통 임베딩 차원을 4배로 늘렸다가 다시 줄임
    """
    # 차원키우기 -> 비선형변환 -> 차원 축소
    def __init__(self, cfg):
        super().__init__()
        # TODO: FeedForward의 확장 배수와 활성화 함수 클래스를 채우세요.
        self.layers = ???????(
            ???????,
            ???????,
            ???????,
        )

    def forward(self, x):
        return ???????



In [4]:
##중요 : 아래는 GPT 모델 전체 구조를 정의하는 코드입니다

#####################################
# Chapter 4: GPT 아키텍처 조립
#####################################
class TransformerBlock(nn.Module):
    """
    표준 트랜스포머 블록 (Decoder Block)
    구조: LayerNorm -> Attention -> Add(Residual) -> LayerNorm -> FeedForward -> Add(Residual)
    """
    def __init__(self, cfg):
        super().__init__()
        self.att = ???????(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = ???????
        self.norm1 = ???????
        self.norm2 = ???????
        self.drop_shortcut = ???????

    def forward(self, x): # 그림을 그대로 구현
        # region [어텐션 블록 (Residual Connection 적용)]
        shortcut = ???????
        x = ???????
        x = ???????
        x = ???????
        x = ???????  #(기울기 소실 방지)
        # endregion

        # region [피드 포워드 블록 (Residual Connection 적용)]
        shortcut = ???????
        x = ???????
        x = ???????
        x = ???????
        x = ???????  # 원본 입력 다시 더함
        # endregion

        return x
class GPTModel(nn.Module):
    """
    전체 GPT 모델 구조 정의
    Embedding -> Transformer Blocks -> Final Norm -> Output Head
    """
    def __init__(self, cfg):
        super().__init__()
        # 토큰 임베딩 (단어 -> 벡터)
        self.tok_emb = ???????
        # 위치 임베딩 (위치 정보 -> 벡터)
        self.pos_emb = ???????
        self.drop_emb = ???????

        # 트랜스포머 블록 쌓기 (n_layers 만큼)
        # *: unpacking operator: 리스트에서 내용물들을 낱개로 꺼내놓은 역할
        # [TransformerBlock(cfg) for _ in range(cfg["n_layers"])]: 트랜스포머 객체 여러개를 list로 만듬
        self.trf_blocks = ???????(
            ???????)

        # 최종 정규화 및 출력 헤드
        self.final_norm = ???????
        # TODO
        self.out_head = ???????

    def forward(self, in_idx): #in_idx = 토큰리스트
        batch_size, seq_len = ???????
        
        # region [임베딩 생성]
        tok_embeds = ???????
        pos_embeds = ???????
        # endregion

        # region [토큰 임베딩과 위치 임베딩 합산]
        x = ???????
        # endregion

        x = ???????
        
        # region [트랜스포머 블록 통과]
        x =???????
        # endregion

        # region [최종 출력 계산]
        x = ???????
        # TODO
        logits = ???????
        # endregion
        
        return logits

In [5]:

def generate_text_simple(model, idx, max_new_tokens, context_size):
    """
    간단한 텍스트 생성 루프
    현재 문맥을 넣어 다음 토큰을 예측하고, 이를 다시 문맥에 추가하여 반복함
    """
    # idx: 현재 문맥의 토큰 인덱스들 (Batch, Time)
    for _ in range(max_new_tokens):

        # 모델이 지원하는 최대 길이(context_size)를 넘지 않도록 자름
        idx_cond = ???????

        # 모델 예측 (기울기 계산 불필요)
        with ???????:
            logits = ???????

        # 마지막 타임스텝의 예측값만 가져옴 (다음 단어 예측이므로)
        # (batch, n_token, vocab_size) -> (batch, vocab_size)
        logits = ???????

        # 가장 확률(로짓값)이 높은 토큰 선택 (Greedy Decoding)
        # TODO
        idx_next = ???????
        
        # 예측된 토큰을 현재 시퀀스 뒤에 이어 붙임
        idx = ???????

    return idx  # 토큰 리스트가 만들어짐


In [6]:
def main():
    # GPT-2 Small 모델 설정값 (124M 파라미터)
    GPT_CONFIG_124M = {
        "vocab_size": 50257,     # 단어 집합 크기
        "context_length": 1024,  # 최대 문맥 길이
        "emb_dim": 768,          # 임베딩 차원
        "n_heads": 12,           # 어텐션 헤드 수
        "n_layers": 12,          # 레이어 수  #트랜스포머블록 수
        "drop_rate": 0.1,        # 드롭아웃 비율
        "qkv_bias": False        # Q,K,V 편향 사용 여부
    }

    torch.???????  # 시드
    
    # 모델 초기화 및 평가 모드 설정 (드롭아웃 비활성화)
    model = ???????
    model.???????()  # 평가모드로 전환. 

    start_context = "Hello, I am"

    # 입력 텍스트 인코딩
    tokenizer = ???????
    encoded = ???????
    encoded_tensor = ???????

    # 텍스트 생성 실행
    out = generate_text_simple(  #out: 출력 토큰id의 리스트
        ???????,
        ???????, #토큰 id의 리스트
        ???????,  # 10개의 새로운 토큰 생성
        ???????
    )
    
    # 생성된 결과 디코딩
    decoded_text = ???????

    print(f"\n\n{50*'='}\n{22*' '}OUT\n{50*'='}")
    print("\nOutput:", out)
    print("Output length:", len(out[0]))
    print("Output text:", decoded_text)

if __name__ == "__main__":
    main()


                      IN

Input text: Hello, I am
Encoded input text: [15496, 11, 314, 716]
encoded_tensor.shape: torch.Size([1, 4])


                      OUT

Output: tensor([[15496,    11,   314,   716, 27018, 24086, 47843, 30961, 42348,  7267,
         49706, 43231, 47062, 34657]])
Output length: 14
Output text: Hello, I am Featureiman Byeswickattribute argue logger Normandy Compton analogous
